# NB06 CLEAN — Musanga satellite abundance and spatial structure (V3.1)

**New reproducible workflow; previous 83-cell NB06 is archived unchanged.** Goal: combine Sentinel-2 seasonal signals (S), 64D annual embeddings (E), and native 4.77 m PlanetScope structure (P) to predict UAV-derived Musanga canopy fraction.

**Research sequence**
1. Reproduce S2-only, E-only, S2+E 50 m baseline from frozen reference cells, forest- and abundance-balanced; distinguish 400 m buffered spatial folds (where R² ~0.6 was obtained) from **strict LOFO**, which was considerably harder.
2. Test spatial context of 64D embeddings with a genuine **2D CNN**, matched against an embedding pointwise baseline on *identical held-out 50 m cells*.
3. Train the **actual UAV-style ResNet18 U-Net** with Planet B/G/R/NIR native 4.77 m patches, comparing train-only global scaling, per-tile scaling and dual reflectance. Model changes documented, not silently substituting the previous small Planet U-Net.
4. Evaluate aggregation onto **common** 10, 30, 50, 75, 100, 125 m grids, not old tile-local cells.
5. Evaluate learned S/E/P branches and all seven input combinations on the same cells/folds, with high-cover bias/recall, equal-forest MAE/R² and calibration.

**Immutable rules:** Baego, MOS (both mosaics held out together), Yangambi, Yoko. Four independent held-out forests. No held-out label leakage; strict reference & footprint comparisons. UAV maps are pseudo-labels, not independent ground truth. Threshold for high cover = 40%. Each stage saves its own inputs, manifests, predictions, models and validation tables.

**Status:** Step 1 runnable below; Steps 2–5 are locked experiment protocols, to be implemented after the baseline is verified. This deliberate division avoids another sprawling experimental notebook.


## 00 — Mount Drive, then locate and validate actual inputs

This cell mounts Google Drive automatically when no known Musanga workspace is visible, **before** searching outputs. If Drive is mounted but caches are missing, it prints an inventory and stops. It never fabricates input files or regenerates imagery.


In [ ]:
# 00 — DRIVE must already be mounted
from pathlib import Path
import numpy as np,pandas as pd,os,json,hashlib
from IPython.display import display
# Original Musanga notebooks accept BOTH workspace locations. Never create an empty
# output folder until an existing source tree is identified.
# 00A — Drive preflight: mounting MUST precede any file inventory.
# Only invoke Colab mount when the expected workspace is not already visible.
KNOWN_FINALS=[Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
              Path('/content/drive/MyDrive/Musanga/Final')]
if not any(f.is_dir() for f in KNOWN_FINALS):
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            'This notebook expects Google Colab with Google Drive connected. '
            'No Musanga workspace is visible in the current Python runtime.') from exc
    print('Musanga workspace not visible: mounting Google Drive now.',flush=True)
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(),(
    'Drive mount did not expose /content/drive/MyDrive. '
    'Inspect Colab Drive authorization and mount status.')
print('Drive mounted:',Path('/content/drive/MyDrive').is_dir())
ROOTS=[Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1'),
       Path('/content/drive/MyDrive/Musanga/Final/Outputs/V3_1')]
SITES=('Baego','MOS','Yangambi','Yoko');KEYS=['site','mosaic','row','col'];HIGH=.4
inventory=[]
for root in ROOTS:
    base=root/'06_spatial_planet_embeddings'
    inventory.append(dict(root=str(root),root_exists=root.is_dir(),
        legacy06_exists=base.is_dir(),
        spatial_file=(base/'10_full_spatial_features_50m.parquet').is_file(),
        target09=(base/'09_uav_targets_50m.parquet').is_file(),
        combined_50m=len(list((base/'11_spatial_ablation_50m/inputs').glob('combined_50m_*.parquet'))),
        legacy_lofo=len(list((base/'11_spatial_ablation_50m/fold_predictions').glob('*/LEAVE_FOREST_OUT_*_B0_S2_EMB.parquet')))))
print('READ-ONLY INPUT INVENTORY (no new extraction):')
display(pd.DataFrame(inventory))
usable=[row for row in inventory if row['combined_50m']>0]
if len(usable)!=1:
    raise FileNotFoundError(
        'Expected exactly one existing 11A satellite cache across known Musanga workspaces. '
        'Cannot reproduce the 50 m baseline without its predictor table. '
        'Copy the input inventory above and identify the workspace if you need help. '
        f'Found {len(usable)} candidates; will not invent or silently regenerate inputs.')
root=Path(usable[0]['root'])
BASE=root/'06_spatial_planet_embeddings'
B11=BASE/'11_spatial_ablation_50m'
OUT=BASE/'CLEAN_NB06';OUT.mkdir(parents=True,exist_ok=True)
spatial_file=BASE/'10_full_spatial_features_50m.parquet'
target09=BASE/'09_uav_targets_50m.parquet'
if spatial_file.is_file():
    reference=pd.read_parquet(spatial_file)
    assert 'target_fraction' in reference
    if 'accepted' in reference:
        reference=reference[reference.accepted.astype(bool)].copy()
    print('Independent source: Section 10 accepted table',spatial_file)
elif target09.is_file():
    reference=pd.read_parquet(target09)
    assert 'mean_probability' in reference
    reference=reference.rename(columns={'mean_probability':'target_fraction'})
    print('Independent source: Section 09 UAV target table',target09)
else:
    reference=None
    print('Neither Section 09 nor Section 10 target file is present.')
    print('Will validate the 11A cache internally (unique keys, finite targets, fixed')
    print('site counts, complete predictor masks) but cannot independently confirm')
    print('its UAV targets against a separate source.')
if reference is not None:
    assert not reference.duplicated(KEYS).any()
    reference=reference.sort_values(KEYS).reset_index(drop=True)
ref={}
for site in SITES:
    matches=sorted((B11/'fold_predictions').glob(
        f'*/LEAVE_FOREST_OUT_{site}_B0_S2_EMB.parquet'))
    if len(matches)==1:
        z=pd.read_parquet(matches[0],columns=KEYS+['observed','predicted'])
        ref[site]=z.sort_values(KEYS).reset_index(drop=True)
    else:
        print('Archive B0 LOFO file not required:',site,'found',len(matches),
              '— using available reference/checks instead')
candidates=sorted((B11/'inputs').glob('combined_50m_*.parquet'));found=[]
for path in candidates:
    d=pd.read_parquet(path)
    if d.duplicated(KEYS).any() or 'target_fraction' not in d:continue
    satellite=[c for c in d if c.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__','EMB_ANNUAL__'))]
    planet=[f'PLANET_{q}_{b}' for q in ('mean','std') for b in ('B','G','R','N')]
    planet += [f'PLANET_texture_{b}' for b in ('NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_highpass')]
    context=[f'EMB_{q}_{j:02d}' for q in ('mean','sd') for j in range(1,65)]
    count=[f'S2_QUARTERLY_3YR__{i:02d}' for i in (11,22,33,44)]
    required=satellite+planet+context
    if not set(required+count).issubset(d.columns):continue
    good=np.isfinite(d[required].to_numpy('float32')).all(1)&(d[count].to_numpy('float32')>=1).all(1)&d.target_fraction.between(0,1).to_numpy()
    d=d.loc[good].sort_values(KEYS).reset_index(drop=True)
    # The authoritative reference does not depend on completed model fits.
    if reference is not None:
        # Section 09 may contain MORE cells than Section 10 (which filtered patches).
        check=reference[reference.set_index(KEYS).index.isin(
            d.set_index(KEYS).index)].reset_index(drop=True)
        agree=(len(check)==len(d) and d[KEYS].equals(check[KEYS])
               and np.allclose(d.target_fraction,check.target_fraction,rtol=0,atol=1e-6))
    else:
        # No independent source: strict 11A coverage and known historical sample count.
        expected={'Baego':480,'MOS':11494,'Yangambi':1383,'Yoko':1097}
        agree=(len(d)==14454 and d.groupby('site').size().to_dict()==expected
               and d[KEYS].notna().all().all() and {'x','y'}.issubset(d.columns)
               and np.isfinite(d[['x','y','target_fraction']].to_numpy(float)).all())
    # If available, also cross-validate the original held-out predictions.
    if agree:
        for forest,b in ref.items():
            a=d.loc[d.site==forest,KEYS+['target_fraction']].reset_index(drop=True)
            if len(a)!=len(b) or not a[KEYS].equals(b[KEYS]) or not np.allclose(
                    a.target_fraction,b.observed,rtol=0,atol=1e-6):
                agree=False
                break
    if agree:found.append((path,d))
assert len(found)==1,('No unique complete cached 11A table passed validation',
    'Candidate files:',[str(p) for p in candidates],
    'Independent reference:',str(spatial_file) if spatial_file.exists() else str(target09) if target09.exists() else 'not available')
SOURCE,D=found[0]
assert len(D)==14454 and {'x','y'}.issubset(D.columns)
FEATURES={'S2':[c for c in D if c.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__'))],
          'EMB':[c for c in D if c.startswith('EMB_ANNUAL__')]}
assert len(FEATURES['EMB'])==64
FEATURES['S2_EMB']=FEATURES['S2']+FEATURES['EMB']
D.to_parquet(OUT/'frozen_50m_reference.parquet',index=False)
print('Frozen',len(D),'matched 50m cells; source:',SOURCE)
print('Reference verification:', 'independent UAV target cross-check' if reference is not None else '11A internal checks only')
print('Predictors:',{k:len(v) for k,v in FEATURES.items()})
display(D.groupby('site').size())





## 01 — Reproducible balanced satellite baseline (S2, E, S2+E)

Use identical 50 m targets and features, three spatial-block folds (400 m, 150 m buffer) **and** four LOFO splits. HGB five-member ensembles, train cap 1500 per forest, target 25% high Musanga ≥40% per selected training forest when available, plus balanced draws among remaining abundance bins. All held-out cells retained. **Do not pool MOS-dominated observations for model selection.**

**Baseline provenance note:** previous ~0.6 spatial R² cannot be assumed for this exact reimplementation. Reproduction is judged by new output, not hard-coded scores.


In [ ]:
# 01A — splits and balanced training selection (no held-out labels in sampling)
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits
threadpool_limits(1)
SEED=20261026;CAP=1500;MEMBERS=5
MAX_NEW_FITS=3   # Smoke-test. Switch to None to complete 105 fits.
PARAM=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,min_samples_leaf=35,
           l2_regularization=2,early_stopping=False)
CFG=dict(version='NB06_CLEAN_S2EMB_v1',source=str(SOURCE),n=len(D),seed=SEED,
         cap=CAP,members=MEMBERS,model=PARAM,feature_lengths={k:len(v) for k,v in FEATURES.items()})
RUN=OUT/'01_satellite'/hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
RUN.mkdir(parents=True,exist_ok=True)
cf=RUN/'config.json'
if cf.exists():assert json.loads(cf.read_text())==CFG
else:cf.write_text(json.dumps(CFG,indent=2))
y=D.target_fraction.to_numpy(float);site=D.site.to_numpy();xy=D[['x','y']].to_numpy(float)
bx=np.floor(xy[:,0]/400).astype('int64');by=np.floor(xy[:,1]/400).astype('int64')
fold=np.mod(bx*73856093+by*19349663,3)
FOLDS=[('SPATIAL',str(i)) for i in range(3)]+[('LOFO',s) for s in SITES]
def split(kind,hold):
    test=(fold==int(hold)) if kind=='SPATIAL' else (site==hold)
    allowed=~test
    if kind=='SPATIAL':
        for forest in SITES:
            it=np.flatnonzero(test&(site==forest));tr=np.flatnonzero(allowed&(site==forest))
            if len(it) and len(tr):
                near=cKDTree(xy[it]).query(xy[tr],k=1)[0]<150
                allowed[tr[near]]=False
    if kind=='LOFO':assert not (site[allowed]==hold).any()
    return np.flatnonzero(allowed),np.flatnonzero(test)
def draw_balanced(pool,seed):
    rng=np.random.default_rng(seed);selected=[]
    for forest in SITES:
        ix=pool[site[pool]==forest]
        if not len(ix):continue
        cap=min(CAP,len(ix))
        if len(ix)<=cap:selected.extend(ix.tolist());continue
        labels=np.digitize(y[ix],[.10,.20,.40]);bins={b:ix[labels==b] for b in range(4)}
        n=min(len(bins[3]),int(np.ceil(cap*.25)))
        pick=rng.choice(bins[3],n,replace=False).tolist() if n else []
        bins[3]=np.setdiff1d(bins[3],pick)
        while len(pick)<cap:
            active=[b for b in bins if len(bins[b])]
            assert active
            for b in active:
                n=min(len(bins[b]),max(1,(cap-len(pick))//len(active)),cap-len(pick))
                if n:
                    take=rng.choice(bins[b],n,replace=False)
                    pick.extend(take.tolist());bins[b]=np.setdiff1d(bins[b],take)
                if len(pick)==cap:break
        selected.extend(pick)
    return np.array(sorted(selected),dtype=int)
print('Ready:',len(FOLDS),'validation folds',len(FEATURES),'predictor families')


In [ ]:
# 01B — checkpointed model fits, fixed fold membership and held-out data
jobs=[(kind,hold,family,m) for kind,hold in FOLDS for family in FEATURES for m in range(MEMBERS)]
new=0
for kind,hold,family,m in jobs:
    train,test=split(kind,hold)
    f=RUN/f'{kind}_{hold}_{family}_m{m}.npz'
    if f.is_file():
        with np.load(f,allow_pickle=False) as z:
            assert np.array_equal(z['test_index'],test) and len(z['pred'])==len(test)
        continue
    if MAX_NEW_FITS is not None and new>=MAX_NEW_FITS:break
    indices=draw_balanced(train,SEED+107*m+sum(map(ord,hold))+len(family)*13)
    X=D[FEATURES[family]].to_numpy('float32')
    estimator=HistGradientBoostingRegressor(random_state=SEED+m,**PARAM)
    estimator.fit(X[indices],y[indices]);pred=np.clip(estimator.predict(X[test]),0,1)
    with open(f.with_suffix('.partial'),'wb') as dest:
        np.savez_compressed(dest,test_index=test,pred=pred.astype('float32'))
    os.replace(f.with_suffix('.partial'),f)
    new+=1;print('Finished',kind,hold,family,'member',m,flush=True)
print('Completed',len(list(RUN.glob('*.npz'))),'of',len(jobs),'fits in',RUN)


In [ ]:
# 01C — final metrics ONLY after all 5 ensemble members exist per case
out=[]
for kind,hold in FOLDS:
    train,test=split(kind,hold)
    for family in FEATURES:
        files=[RUN/f'{kind}_{hold}_{family}_m{m}.npz' for m in range(MEMBERS)]
        if not all(f.exists() for f in files):continue
        preds=[]
        for f in files:
            with np.load(f,allow_pickle=False) as z:
                assert np.array_equal(z['test_index'],test)
                preds.append(z['pred'].astype(float))
        prediction=np.mean(preds,axis=0)
        for forest in SITES:
            mask=site[test]==forest
            if not mask.any():continue
            a=y[test][mask];b=prediction[mask];e=b-a;high=a>=HIGH
            out.append(dict(scheme=kind,holdout=hold,model=family,forest=forest,n=len(a),
                 MAE_pp=100*np.mean(np.abs(e)),R2=1-np.sum(e**2)/np.sum((a-a.mean())**2) if np.var(a)>0 else np.nan,
                 bias_pp=100*e.mean(),high_n=int(high.sum()),
                 high_bias_pp=100*e[high].mean() if high.any() else np.nan,
                 high_recall=np.mean(b[high]>=HIGH) if high.any() else np.nan,
                 mean_target=a.mean(),target_var=a.var(),mse=np.mean(e*e)))
RES=pd.DataFrame(out)
assert len(RES)>0,'No complete ensemble yet; continue 01B with MAX_NEW_FITS=None'
print('Per-forest results; LOFO contains exactly one held-out forest each:')
display(RES.round(3))
rows=[]
for (scheme,hold,model),g in RES.groupby(['scheme','holdout','model']):
    u=g.mean_target.mean();var=np.mean(g.target_var+(g.mean_target-u)**2)
    rows.append(dict(scheme=scheme,holdout=hold,model=model,n_forests=len(g),
                     equal_forest_MAE_pp=g.MAE_pp.mean(),equal_forest_R2=1-g.mse.mean()/var,
                     mean_forest_R2=g.R2.mean()))
SUMMARY=pd.DataFrame(rows)
display(SUMMARY.round(3))
RES.to_csv(RUN/'validation_by_forest.csv',index=False)
SUMMARY.to_csv(RUN/'validation_summary.csv',index=False)
print('Saved',RUN)


## 01D — Paired observed–predicted scatterplots and *actual* abundance-balance audit

**Plots:** S2, EMB and S2+EMB in three rows; buffered spatial-block CV and strict LOFO in two columns. All predictions are reconstructed from the **five saved ensemble-member NPZ files**. Each test cell appears **exactly once per model and validation scheme**, and the same stratified display subsample is used for all models (up to 350 test cells per forest). Display sampling is **only for the graphic**; the statistics use **every held-out cell**, with equal-forest weights (not MOS-dominated pooled metrics). Each scatterplot includes the 1:1 line, a forest-balanced binned response curve, equal-forest R² / MAE, and a shared 0–100% axis.

**Abundance audit:** reproduce the exact deterministic `draw_balanced()` selections that trained each ensemble member, then compare full eligible training pools with the actual sampled training sets across the four abundance strata (`<10%`, `10–20%`, `20–40%`, `≥40%`). The sampler balances forests via per-forest caps and *increases representation of high-abundance cells where available*; it does **not guarantee equal numbers in every abundance bin** (particularly for forests below the cap). Show both aggregate and **per-training-forest** percentages. This does not re-fit models or require additional imagery.

Run after **01C** in the same Colab session. PNG + PDF figures and CSV tables are saved beneath `RUN / 'figures'`.


In [ ]:
# 01D — reproducible paired scatterplot and actual training-abundance audit; CPU only
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

FIG01D = RUN / 'figures'
FIG01D.mkdir(parents=True, exist_ok=True)
MODEL_ORDER = ['S2', 'EMB', 'S2_EMB']
FOREST_ORDER = list(SITES)
COLOR01D = dict(zip(FOREST_ORDER, ['#3575B7', '#D18B31', '#2B9D80', '#9A63A8']))
MAX_DISPLAY_PER_FOREST = 350
ABUNDANCE_EDGES = np.array([0, .10, .20, .40, 1.0000001])
ABUNDANCE_NAMES = ['0–10%', '10–20%', '20–40%', '≥40%']

# Reconstruct exactly the held-out ensemble mean used by 01C.
frames = []
for scheme in ('SPATIAL', 'LOFO'):
    blocks = [x for x in FOLDS if x[0] == scheme]
    for _, holdout in blocks:
        train_ix, test_ix = split(scheme, holdout)
        for model in MODEL_ORDER:
            sources = [RUN / f'{scheme}_{holdout}_{model}_m{m}.npz'
                       for m in range(MEMBERS)]
            assert all(p.is_file() for p in sources), (
                'Complete 01B before plotting', scheme, holdout, model,
                [str(p) for p in sources if not p.is_file()])
            members = []
            for source in sources:
                with np.load(source, allow_pickle=False) as z:
                    assert np.array_equal(z['test_index'], test_ix), (
                        'Prediction/holdout mismatch', source)
                    members.append(z['pred'].astype('float32'))
            pred = np.mean(np.stack(members), axis=0)
            frames.append(pd.DataFrame({
                'scheme': scheme, 'holdout': str(holdout),
                'model': model, 'row_id': test_ix,
                'forest': site[test_ix], 'observed': y[test_ix],
                'predicted': pred
            }))
PLOT01D = pd.concat(frames, ignore_index=True)
assert np.isfinite(PLOT01D[['observed', 'predicted']].to_numpy()).all()
for scheme in ('SPATIAL', 'LOFO'):
    for model in MODEL_ORDER:
        g = PLOT01D[(PLOT01D.scheme == scheme) & (PLOT01D.model == model)]
        assert len(g) == len(D) and g.row_id.nunique() == len(D), (
            'Expected each reference cell exactly once per validation scheme',
            scheme, model, len(g), g.row_id.nunique(), len(D))
        assert np.array_equal(np.sort(g.row_id.to_numpy()), np.arange(len(D)))

# Forest-equal metrics on ALL test cells (unbalanced MOS counts do not dominate).
rows01d = []
for (scheme, model), g in PLOT01D.groupby(['scheme', 'model'], sort=False):
    per_forest = []
    for forest in FOREST_ORDER:
        a = g.loc[g.forest == forest, 'observed'].to_numpy(float)
        b = g.loc[g.forest == forest, 'predicted'].to_numpy(float)
        e = b - a
        per_forest.append(dict(forest=forest, n=len(a),
            mean_target=a.mean(), target_variance=a.var(),
            MSE=np.square(e).mean(), MAE_pp=100*np.abs(e).mean(),
            R2=(1-np.square(e).sum()/np.square(a-a.mean()).sum()
                if a.var() > 0 else np.nan)))
    f = pd.DataFrame(per_forest).set_index('forest').loc[FOREST_ORDER]
    mean_y = f.mean_target.mean()
    forest_equal_variance = np.mean(
        f.target_variance.to_numpy() + (f.mean_target.to_numpy()-mean_y)**2)
    rows01d.append(dict(scheme=scheme, model=model,
        equal_forest_R2=1-f.MSE.mean()/forest_equal_variance,
        equal_forest_MAE_pp=f.MAE_pp.mean(),
        mean_forest_R2=f.R2.mean(),
        n_test=len(g)))
METRICS01D = pd.DataFrame(rows01d)
print('Full-test-cell metrics (not plotted subset):')
display(METRICS01D.round(3))

# Same display cells for ALL models; fixed number from each forest.
rng = np.random.default_rng(20261026)
display_ids = []
for forest in FOREST_ORDER:
    candidates = np.flatnonzero(site == forest)
    count = min(MAX_DISPLAY_PER_FOREST, len(candidates))
    display_ids.extend(rng.choice(candidates, count, replace=False).tolist())
display_ids = np.asarray(sorted(display_ids), dtype=int)
fig, axs = plt.subplots(3, 2, figsize=(12.6, 16.0),
                         sharex=True, sharey=True, constrained_layout=True)
for row, model in enumerate(MODEL_ORDER):
    for col, scheme in enumerate(('SPATIAL', 'LOFO')):
        ax = axs[row, col]
        g = PLOT01D[(PLOT01D.scheme == scheme) &
                    (PLOT01D.model == model)].set_index('row_id')
        sub = g.loc[display_ids]
        for forest in FOREST_ORDER:
            f = sub[sub.forest == forest]
            ax.scatter(100*f.observed, 100*f.predicted,
                       s=12, alpha=.32, rasterized=True,
                       color=COLOR01D[forest], label=forest)
        ax.plot([0, 100], [0, 100], linestyle='--', color='black',
                linewidth=1, alpha=.65)
        # Show a forest-balanced response by observed abundance, using all
        # held-out cells, NOT the display-only sampling.
        bins = np.arange(0, 1.00001, .10)
        by_forest=[]
        for forest in FOREST_ORDER:
            a = g[g.forest == forest]
            k = np.minimum(np.digitize(a.observed, bins, right=False)-1, 9)
            for q in range(10):
                z = a.iloc[np.flatnonzero(k == q)]
                if len(z):
                    by_forest.append((q, forest, float(z.observed.mean()),
                                      float(z.predicted.mean())))
        curve = pd.DataFrame(by_forest, columns=['bin','forest','obs','pred'])
        line = curve.groupby('bin', as_index=False)[['obs','pred']].mean()
        ax.plot(100*line.obs, 100*line.pred, 'o-', color='#171717',
                markersize=4, linewidth=2, label='Forest-equal binned mean')
        metric = METRICS01D[(METRICS01D.scheme == scheme) &
                            (METRICS01D.model == model)].iloc[0]
        ax.set_title(f'{model} — {scheme} holdout\\n'
            f"equal-forest R²={metric.equal_forest_R2:.3f}  "
            f"MAE={metric.equal_forest_MAE_pp:.2f} pp",
            fontsize=11)
        ax.set_xlim(0,100); ax.set_ylim(0,100)
        ax.grid(alpha=.12)
        if col==0: ax.set_ylabel('Predicted Musanga cover (%)')
        if row==2: ax.set_xlabel('Observed UAV-derived Musanga cover (%)')
handles,labels=axs[0,0].get_legend_handles_labels()
fig.legend(handles,labels,loc='outside lower center',ncol=5,fontsize=9)
fig.suptitle('Satellite abundance: buffered spatial CV versus forest transfer\\n'
             'Matched 50 m reference cells; equal-forest visualization',
             fontsize=14)
png01d=FIG01D/'01D_spatial_vs_LOFO_obs_pred.png'
pdf01d=FIG01D/'01D_spatial_vs_LOFO_obs_pred.pdf'
fig.savefig(png01d,dpi=200,bbox_inches='tight')
fig.savefig(pdf01d,bbox_inches='tight')
plt.show();plt.close(fig)

# Audit training populations and the EXACT member-specific sampled indices.
# NO test labels used to choose training examples.
audit=[]
for scheme,hold in FOLDS:
    eligible,heldout=split(scheme,hold)
    for member in range(MEMBERS):
        for model in MODEL_ORDER:
            indices=draw_balanced(
                eligible, SEED+107*member+sum(map(ord,hold))+len(model)*13)
            for population,ix in [('Eligible pool',eligible),
                                  ('Actual selected',indices)]:
                for forest in FOREST_ORDER:
                    values=y[ix[site[ix]==forest]]
                    if not len(values):continue
                    classes=np.minimum(np.digitize(values,ABUNDANCE_EDGES)-1,3)
                    counts=np.bincount(classes,minlength=4)
                    for j,name in enumerate(ABUNDANCE_NAMES):
                        audit.append(dict(scheme=scheme,holdout=str(hold),
                            model=model,member=member,forest=forest,
                            population=population,bin=name,n=int(counts[j]),
                            forest_total=len(values),
                            fraction=float(counts[j]/len(values))))
AUDIT01D=pd.DataFrame(audit)
# Avoid counting the same eligible reference population repeatedly across
# the five ensemble members/models when summarizing distributions.
audit_summary = (AUDIT01D.groupby(
    ['scheme','population','bin'],as_index=False)
    .agg(mean_forest_fraction=('fraction','mean')))
print('Abundance by training scheme and population (mean across available forests / folds / members):')
display(audit_summary.pivot_table(index=['scheme','population'],
    columns='bin',values='mean_forest_fraction').mul(100).round(1))
print('Actual sampled training proportions for each training forest — LOFO folds:')
per_site01d = (AUDIT01D[(AUDIT01D.scheme=='LOFO') &
                        (AUDIT01D.population=='Actual selected')]
               .groupby(['holdout','forest','bin'],as_index=False)
               .agg(mean_percent=('fraction',lambda x:100*x.mean())))
display(per_site01d.pivot_table(index=['holdout','forest'],
    columns='bin',values='mean_percent').round(1))

# Abundance figure: equal-weighted mean forest share, not absolute pixel counts.
f,ax=plt.subplots(figsize=(9,4.7),layout='constrained')
selected=audit_summary[audit_summary.scheme=='LOFO']
labels=['Eligible pool','Actual selected']
xx=np.arange(2)
bottom=np.zeros(2)
for j,name in enumerate(ABUNDANCE_NAMES):
    vals=np.array([float(selected[(selected.population==p)&
                        (selected.bin==name)].mean_forest_fraction.iloc[0])
                    for p in labels])*100
    ax.bar(xx,vals,bottom=bottom,label=name)
    bottom+=vals
ax.set_xticks(xx,labels)
ax.set_ylim(0,100);ax.set_ylabel('Mean abundance-bin share within forest (%)')
ax.set_title('LOFO: eligible vs actually selected training-cell abundance')
ax.legend(title='UAV-derived cover',bbox_to_anchor=(1.02,1),loc='upper left')
balance_png=FIG01D/'01D_abundance_balance_LOFO.png'
f.savefig(balance_png,dpi=200,bbox_inches='tight')
plt.show();plt.close(f)
METRICS01D.to_csv(FIG01D/'01D_full_support_metrics.csv',index=False)
AUDIT01D.to_csv(FIG01D/'01D_abundance_actual_draws.csv',index=False)
PLOT01D.to_parquet(FIG01D/'01D_all_cell_holdout_predictions.parquet',index=False)
print('01D figures:',png01d,pdf01d,balance_png,sep='\\n  ')
print('No fitting performed. Display-only sampling does not affect metrics.')



## 02A — Native 10 m Earth-embedding spatial neighborhoods (cached only)

**Input:** the same frozen 14,454 50 m reference cell centers as Stage 1 and four annual 64D embedding rasters already extracted in NB04. **No new Earth Engine request.** Extract **9×9 pixels ×64 bands** (90×90 m context at the actual 10 m resolution), centered on each 50 m sample center, with CRS-aware positioning and source valid-data masks. One reusable indexed memmap cache, stable config, progress checkpoint and source-file checks. Source raster coordinates are never inferred from the 50 m row/column numbers.

**Important:** a 50 m target cell overlaps multiple native 10 m embedding pixels. Stage 2 compares the 2D model with a *center-only ablation using the identical CNN network*. The pre-existing HGB embedding model is an additional operational baseline, not an architecture-matched control.

Run **02A once** after 00/01A; it needs no GPU. Source data remain read-only.


In [ ]:
# 02A — Cache 64x9x9 native embedding context per frozen 50 m reference cell.
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from rasterio.warp import transform as crs_transform
from IPython.display import display
EMB_SOURCE02 = root/'04_satellite_predictors/input_layers/EMB_ANNUAL'
YEARS02={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
P02=9
EMBFILES02={s:EMB_SOURCE02/f'{s}_EMB_ANNUAL_{YEARS02[s]}_10m.tif' for s in SITES}
for site02,f in EMBFILES02.items():
    assert f.is_file(),('Missing annual embedding raster',site02,f)
CONFIG02A=dict(version='clean_stage2_native_embedding_9x9_v1',
   source50=str(SOURCE),n_cells=len(D),patch=P02,
   source={k:dict(path=str(f),size=f.stat().st_size,
                 mtime_ns=f.stat().st_mtime_ns) for k,f in EMBFILES02.items()})
KEY02A=hashlib.sha256(json.dumps(CONFIG02A,sort_keys=True).encode()).hexdigest()[:16]
CACHE02=OUT/'02_embedding_context'/KEY02A
CACHE02.mkdir(parents=True,exist_ok=True)
cfg02=CACHE02/'config.json'
if cfg02.exists():assert json.loads(cfg02.read_text())==CONFIG02A
else:cfg02.write_text(json.dumps(CONFIG02A,indent=2))
ARR02=CACHE02/'embedding_64x9x9.npy'
MASK02=CACHE02/'valid_9x9.npy'
STATUS02=CACHE02/'complete_indices.npy'
# Initialize restartable arrays only once; avoid holding all spatial patches in RAM.
if not ARR02.exists():
    arr02=np.lib.format.open_memmap(ARR02,mode='w+',dtype='float16',
                                  shape=(len(D),64,P02,P02))
    arr02.flush();del arr02
if not MASK02.exists():
    valid02=np.lib.format.open_memmap(MASK02,mode='w+',dtype='uint8',
                                    shape=(len(D),P02,P02))
    valid02.flush();del valid02
if not STATUS02.exists():
    done02=np.lib.format.open_memmap(STATUS02,mode='w+',dtype='uint8',shape=(len(D),))
    done02[:]=0;done02.flush();del done02
patch02=np.load(ARR02,mmap_mode='r+')
valid02=np.load(MASK02,mmap_mode='r+')
done02=np.load(STATUS02,mmap_mode='r+')
assert patch02.shape==(len(D),64,P02,P02)
assert valid02.shape==(len(D),P02,P02)
assert done02.shape==(len(D),)
for forest02 in SITES:
    ids02=np.flatnonzero(site==forest02)
    pending02=ids02[done02[ids02]==0]
    if not len(pending02):
        print('CACHED 02A:',forest02,len(ids02));continue
    with rasterio.open(EMBFILES02[forest02]) as src02:
        assert src02.count==64 and abs(src02.res[0]-10)<.1
        positions02=xy[pending02]
        sourcecrs02=str(D.crs.iloc[pending02[0]]) if 'crs' in D else 'EPSG:32635'
        if sourcecrs02!=str(src02.crs):
            xtrans02,ytrans02=crs_transform(sourcecrs02,src02.crs,
                            positions02[:,0].tolist(),positions02[:,1].tolist())
            positions02=np.column_stack([xtrans02,ytrans02])
        for j,(k,(x02,y02)) in enumerate(zip(pending02,positions02)):
            col02,row02=(~src02.transform)*(float(x02),float(y02))
            col02=int(np.floor(col02));row02=int(np.floor(row02))
            tile02=src02.read(window=Window(col02-4,row02-4,P02,P02),
                     boundless=True,masked=True,out_dtype='float32')
            good02=~np.ma.getmaskarray(tile02).any(axis=0)
            data02=np.ma.filled(tile02,0).astype('float32')
            good02 &= np.isfinite(data02).all(axis=0)
            assert good02.sum()>=25,('Too little native embedding context',forest02,int(k),int(good02.sum()))
            data02[:,~good02]=0
            patch02[k]=data02.astype('float16')
            valid02[k]=good02.astype('uint8')
            done02[k]=1
            if (j+1)%250==0:
                patch02.flush();valid02.flush();done02.flush()
                print('02A',forest02,j+1,'/',len(pending02),flush=True)
    patch02.flush();valid02.flush();done02.flush()
    print('02A completed',forest02,'cells',len(ids02),flush=True)
assert np.all(done02==1),('Incomplete native embedding cache',int(np.sum(done02)),len(D))
print('02A native 10 m embeddings ready:',CACHE02)
display(pd.DataFrame([dict(site=s,n=int((site==s).sum()),
       mean_valid_fraction=float(valid02[site==s].mean())) for s in SITES]))


## 02B — Controlled 2D embedding CNN versus center-only embedding CNN

**Two equally sized models, same samples, same model architecture and same 50 m target cells:**
- `CENTER`: only the center 64-dimensional embedding vector is available, repeated across the 9×9 patch (a control for parameter count and architecture).
- `SPATIAL`: the genuine 64×9×9 embedding neighborhood with its arrangement intact. Small spatial CNN encodes local structure.

Train-only band standardization, **identical forest- and abundance-balanced draws from Stage 1**, masked pixel inputs, strict 400 m buffered SPATIAL folds and 4-forest LOFO. Repeat **five independent members** to compare ensemble means fairly to existing Stage 1 HGB five-member ensembles. Fresh GPU sessions are supported via read-only .npy memmap cache and restartable .npz predictions. One fold/variant/member fit per invocation by default; set `MAX_NEW_FITS02B=None` for all 70 fits.

**Interpretation warning:** A 9×9 patch is 90×90 m; neighboring 50 m targets have overlapping embedding patches. Buffered spatial CV excludes nearby *training centers*, but patches can still overlap beyond the buffer boundaries if all edges are not excluded. The 150 m buffer exceeds a 90 m patch diameter and normally prevents train/test patch overlap. Forest LOFO is fully geographically independent.


In [ ]:
# 02B — 2D embedding spatial CNN vs 64D-center-only controlled CNN.
from pathlib import Path
import os,json,hashlib,random,time
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from IPython.display import display
assert 'CACHE02' in globals() and 'RUN' in globals() and 'draw_balanced' in globals()
assert np.all(np.load(CACHE02/'complete_indices.npy',mmap_mode='r')==1), 'Complete 02A first'
DEVICE02=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED02=20261026
EPOCHS02=25
BATCH02=128
LR02=.0006
MEMBERS02=5
MAX_NEW_FITS02B=1  # Smoke test; set None to complete both variants × 7 folds × 5 members.
VARIANTS02=('CENTER','SPATIAL')
CONF02=dict(version='NB06_emb_2d_spatial_control_v1',input_cache=CACHE02.name,
   seed=SEED02,epochs=EPOCHS02,batch=BATCH02,lr=LR02,members=MEMBERS02,
   forest_balanced=True,abundance_sampling='identical_stage01_draw_balanced',
   variants=list(VARIANTS02),patch='64x9x9_native10m',
   teacher='frozen_matched_50m_UAV_probability')
RUN02=OUT/'02_embedding_context_CNN'/hashlib.sha256(
   json.dumps(CONF02,sort_keys=True).encode()).hexdigest()[:16]
RUN02.mkdir(parents=True,exist_ok=True)
cc02=RUN02/'config.json'
if cc02.exists():assert json.loads(cc02.read_text())==CONF02
else:cc02.write_text(json.dumps(CONF02,indent=2))
torch.set_num_threads(2)
X02=np.load(CACHE02/'embedding_64x9x9.npy',mmap_mode='r')
VM02=np.load(CACHE02/'valid_9x9.npy',mmap_mode='r')
assert X02.shape==(len(D),64,9,9) and VM02.shape==(len(D),9,9)

def training_norm02(train_idx):
    # Only train-forest samples contribute. Robust per-band pixel distribution.
    selected=train_idx[::max(1,len(train_idx)//700)]
    chunk=np.asarray(X02[selected],dtype='float32')
    valid=np.asarray(VM02[selected],dtype=bool)
    sample=chunk.transpose(1,0,2,3).reshape(64,-1)
    ok=valid.ravel()
    data=sample[:,ok]
    mid=np.median(data,axis=1)
    scale=np.maximum((np.percentile(data,90,axis=1)-
                      np.percentile(data,10,axis=1))/2,.001)
    assert np.isfinite(mid).all() and np.isfinite(scale).all()
    return mid.astype('float32'),scale.astype('float32')

class EmbTiles02(Dataset):
    def __init__(self,ids,variant,norm,augment):
        self.ids=np.asarray(ids,dtype=int)
        self.variant=variant
        self.mid,self.scale=[z[:,None,None] for z in norm]
        self.augment=augment
    def __len__(self):return len(self.ids)
    def __getitem__(self,j):
        idx=int(self.ids[j])
        x=np.array(X02[idx],dtype='float32',copy=True)
        valid=np.asarray(VM02[idx],dtype=bool)
        if self.variant=='CENTER':
            x=np.broadcast_to(x[:,4:5,4:5],(64,9,9)).copy()
            # Control uses identical 2D conv network but no neighboring features.
            valid=np.full((9,9),bool(valid[4,4]))
        x=np.clip((x-self.mid)/self.scale,-3,3)
        x[:,~valid]=0
        if self.augment:
            if random.random()<.5:x=x[:,:,::-1]
            if random.random()<.5:x=x[:,::-1,:]
            x=np.rot90(x,random.randrange(4),axes=(1,2))
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.tensor(y[idx],dtype=torch.float32))

class EmbCNN02(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat=nn.Sequential(
            nn.Conv2d(64,32,3,padding=1),nn.ReLU(inplace=True),
            nn.Conv2d(32,32,3,padding=1),nn.ReLU(inplace=True),
            nn.Conv2d(32,32,3,padding=1),nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),nn.Flatten())
        self.head=nn.Sequential(nn.Linear(32,32),nn.ReLU(inplace=True),
                                nn.Linear(32,1),nn.Sigmoid())
    def forward(self,x):return self.head(self.feat(x)).squeeze(1)

def fit02(scheme,hold,variant,member,train,test):
    seed=SEED02+sum(map(ord,hold))+member*107
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    # SAME selected training row ids for both variants and Stage 1 HGB member.
    selected=draw_balanced(train,SEED+107*member+sum(map(ord,hold))+
                           len('EMB')*13)
    norm=training_norm02(train)
    train_loader=DataLoader(EmbTiles02(selected,variant,norm,True),
         batch_size=BATCH02,shuffle=True,num_workers=0)
    test_loader=DataLoader(EmbTiles02(test,variant,norm,False),
         batch_size=BATCH02,shuffle=False,num_workers=0)
    net=EmbCNN02().to(DEVICE02)
    optim=torch.optim.AdamW(net.parameters(),lr=LR02,weight_decay=.001)
    for epoch in range(EPOCHS02):
        net.train();sq=0;n=0
        for x,z in train_loader:
            x=x.to(DEVICE02);z=z.to(DEVICE02)
            optim.zero_grad(set_to_none=True)
            yp=net(x);loss=torch.square(yp-z).mean()
            loss.backward();optim.step()
            sq+=float(loss.item())*len(z);n+=len(z)
        if epoch==0 or (epoch+1)%5==0:
            print(f'02B {scheme} {hold} {variant} m{member} epoch {epoch+1}/{EPOCHS02} MSE={sq/n:.5f}',flush=True)
    net.eval();pred=[]
    with torch.inference_mode():
        for x,z in test_loader:
            pred.append(net(x.to(DEVICE02)).cpu().numpy().astype('float32'))
    return np.concatenate(pred),net,norm,selected

new02=0
for scheme,hold in FOLDS:
    tr,te=split(scheme,hold)
    for variant in VARIANTS02:
        for member in range(MEMBERS02):
            dest=RUN02/f'{scheme}_{hold}_{variant}_m{member}.npz'
            if dest.is_file():
                with np.load(dest,allow_pickle=False) as z:
                    assert np.array_equal(z['test_index'],te)
                    assert len(z['pred'])==len(te)
                continue
            if MAX_NEW_FITS02B is not None and new02>=MAX_NEW_FITS02B:break
            assert DEVICE02.type=='cuda','GPU required for new 02B fitting'
            start=time.monotonic()
            pred,model,norm,selected=fit02(scheme,hold,variant,member,tr,te)
            partial=dest.with_suffix('.partial')
            with open(partial,'wb') as f:
                np.savez_compressed(f,test_index=te.astype('int32'),pred=pred)
            os.replace(partial,dest)
            torch.save(dict(state=model.state_dict(),config=CONF02,
                 norm_median=norm[0],norm_scale=norm[1],
                 selected_training_index=selected),
                 RUN02/f'{scheme}_{hold}_{variant}_m{member}.pt')
            new02+=1
            print('02B SAVED',scheme,hold,variant,member,
                  'seconds',round(time.monotonic()-start,1),flush=True)
        if MAX_NEW_FITS02B is not None and new02>=MAX_NEW_FITS02B:break
    if MAX_NEW_FITS02B is not None and new02>=MAX_NEW_FITS02B:break
print('02B outputs',RUN02,'completed',len(list(RUN02.glob('*.npz'))),
      'of',len(FOLDS)*len(VARIANTS02)*MEMBERS02)



## 02 — Embedding spatial context (next implementation gate)

**Fixed test:** pointwise annual 64D embeddings (Stage 1) vs genuine 2D patches (e.g. 7×7 cells on the *native 10 m embedding lattice*) through a compact CNN; same frozen 50 m cell centers and LOFO/spatial folds. The 2D model must use true geographic neighborhood placement, valid-data masks, **train-forest-only normalization**, and no target leakage. Compare E, E+neighborhood summary, E+2D CNN, SE.

**Do not merely resize 10 m embeddings to 4.77 m and claim higher spatial resolution.** Report improvement on exactly paired test cells and the ability to identify high-cover MOS.


## 03 — PlanetScope 4.77 m ResNet18 U-Net (next implementation gate)

Use 422 verified 64×64 native Planet B/G/R/NIR tiles and dense UAV teacher maps. **Recover the exact UAV ResNet18 U-Net encoder and decoder configuration** from the successful UAV notebook/checkpoint before implementation; the previous small Planet U-Net is not the same model. Adapt the input stem to 4 bands while documenting initialization.

Controlled input treatments: robust global train-forest normalization; tile-relative robust normalization; dual absolute+relative bands. Moderate abundance-balanced, forest-balanced tile sampling; masked dense regression; strict four LOFO. Compare 4.77 m discrimination and high-abundance performance. Keep Planet annual only.


## 04 — Aggregation on common geographic support (next implementation gate)

**Evaluate rather than retrain first:** reproject dense 4.77 m Planet predictions, S2 predictions and embedding predictions onto identical predeclared 10/30/50/75/100/125 m grids. Use area-weighted aggregation, explicit coverage fractions and one exact reference UAV mask for all candidates. Exclude unsupported edge cells consistently. No tile-local 50 m comparison to S2's different grid.

Report per forest and equal forest: R², MAE, RMSE, bias, Pearson r, high-cover bias and recall, and balanced scatterplots.


## 05 — Learned S/E/P fusion and seven ablations (final stage)

After each independent component has passed its validation gate, test S, E, P, SE, SP, EP, SEP under **the same four LOFO held-out forests, identical geographic footprints and reference targets**. S2 supplies seasonality, embeddings supply 2D context, Planet supplies 4.77 m canopy structure. Feature fusion weights are learned; avoid forcing hand-picked contributions. Select on equal-forest accuracy AND MOS high-cover recall/bias, not just Yangambi's low prevalence. Four forests are four independent transfer tests, not thousands of independent field sites.

**Do not add experimental cells above these locked stage contracts.** Implement one self-contained, cached stage at a time after the previous stage is verified.
